In [0]:
# Check PySpark is available
spark

In [0]:
base_path = "/Volumes/workspace/dinedash/raw_data"

print(base_path)

In [0]:
display(dbutils.fs.ls(base_path))

In [0]:
dimensions_path = f"{base_path}/dimensions"

display(dbutils.fs.ls(dimensions_path))

In [0]:
customers_path = f"{dimensions_path}/dim_customers.csv"

df_customers = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(customers_path)
)

display(df_customers.limit(5))

In [0]:
df_customers.printSchema()

In [0]:
df_customers.count()

In [0]:
df_delivery_agents = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{dimensions_path}/dim_delivery_agents.csv")
)

df_locations = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{dimensions_path}/dim_locations.csv")
)

df_menu_items = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{dimensions_path}/dim_menu_items.csv")
)

df_restaurants = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{dimensions_path}/dim_restaurants.csv")
)

In [0]:
print("Customers:", df_customers.count())
print("Delivery Agents:", df_delivery_agents.count())
print("Locations:", df_locations.count())
print("Menu Items:", df_menu_items.count())
print("Restaurants:", df_restaurants.count())

In [0]:
from pyspark.sql.functions import col, sum

df_customers.select(
    *[
        sum(col(column).isNull().cast("int")).alias(column)
        for column in df_customers.columns
    ]
).display()

In [0]:
df_customers.filter(
    col("email").isNull() | col("location_id").isNull()
).display()

In [0]:
print("Total rows:", df_customers.count())
print("Distinct rows:", df_customers.distinct().count())

In [0]:
duplicate_customers = (
    df_customers
    .groupBy(df_customers.columns)
    .count()
    .filter("count > 1")
)

duplicate_customers.show()

In [0]:
df_customers_clean = df_customers.dropDuplicates()

In [0]:
print("Before removing duplicates:", df_customers.count())
print("After removing duplicates:", df_customers_clean.count())

In [0]:
from pyspark.sql.functions import col

missing_data = df_customers_clean.filter(
    col("email").isNull() | col("location_id").isNull()
)

missing_data.show()

In [0]:
from pyspark.sql.functions import col, sum

df_customers_clean.select(
    *[
        sum(col(column).isNull().cast("int")).alias(column)
        for column in df_customers_clean.columns
    ]
).show()

In [0]:
from pyspark.sql.functions import col

df_customers_clean.filter(
    col("email").isNull() & col("location_id").isNull()
).count()

In [0]:
df_customers_clean.filter(
    col("email").isNull() & col("location_id").isNull()
).show(50, truncate=False)

In [0]:
from pyspark.sql.functions import count, col

duplicate_customer_ids = (
    df_customers_clean
    .groupBy("customer_id")
    .agg(count("*").alias("count"))
    .filter(col("count") > 1)
)

duplicate_customer_ids.show()

In [0]:
# Check whether all non-null customer location_ids exist in df_locations

invalid_locations = (
    df_customers_clean
    .filter(col("location_id").isNotNull())
    .select("location_id")
    .join(
        df_locations.select("location_id"),
        on="location_id",
        how="left_anti"
    )
)

print("Invalid location IDs:", invalid_locations.count())

invalid_locations.show()

In [0]:
print("Total rows:", df_delivery_agents.count())
print("Distinct rows:", df_delivery_agents.distinct().count())

In [0]:
duplicate_delivery_agents = df_delivery_agents.groupBy(
    df_delivery_agents.columns
).count().filter("count > 1")

print("Duplicate groups:", duplicate_delivery_agents.count())

duplicate_delivery_agents.show(50, truncate=False)

In [0]:
df_delivery_agents_clean = df_delivery_agents.dropDuplicates()

print("Before removing duplicates:", df_delivery_agents.count())
print("After removing duplicates:", df_delivery_agents_clean.count())

In [0]:
from pyspark.sql.functions import col, sum

df_delivery_agents_clean.select(
    [
        sum(col(column).isNull().cast("int")).alias(column)
        for column in df_delivery_agents_clean.columns
    ]
).show()

In [0]:
from pyspark.sql.functions import col, count

duplicate_agent_ids = (
    df_delivery_agents_clean
    .groupBy("agent_id")
    .count()
    .filter(col("count") > 1)
)

print("Duplicate agent IDs:", duplicate_agent_ids.count())

duplicate_agent_ids.show()

In [0]:
df_delivery_agents_clean.select(
    "rating"
).summary().show()

In [0]:
from pyspark.sql.functions import col

invalid_ratings = df_delivery_agents_clean.filter(
    (col("rating") < 0) | (col("rating") > 5)
)

print("Invalid ratings:", invalid_ratings.count())

invalid_ratings.show()

In [0]:
df_delivery_agents_clean.filter(
    col("rating") == 0
).show(truncate=False)

In [0]:
missing_phone_and_zero_rating = df_delivery_agents_clean.filter(
    col("phone").isNull() & (col("rating") == 0)
).count()

print("Agents with both missing phone and 0 rating:", missing_phone_and_zero_rating)

In [0]:
zero_rating = df_delivery_agents_clean.filter(
    col("rating") == 0
).count()

print("Total agents with 0 rating:", zero_rating)

In [0]:
df_delivery_agents_clean.filter(
    col("phone").isNotNull()
).select("phone").show(truncate=False)

In [0]:
print("Total rows:", df_locations.count())
print("Distinct rows:", df_locations.distinct().count())

In [0]:
df_locations.select(
    *[
        sum(col(column).isNull().cast("int")).alias(column)
        for column in df_locations.columns
    ]
).show()

In [0]:
duplicate_location_ids = (
    df_locations
    .groupBy("location_id")
    .count()
    .filter(col("count") > 1)
)

print("Duplicate location IDs:", duplicate_location_ids.count())

In [0]:
invalid_coordinates = df_locations.filter(
    (col("latitude") < -90) |
    (col("latitude") > 90) |
    (col("longitude") < -180) |
    (col("longitude") > 180)
)

print("Invalid coordinate records:", invalid_coordinates.count())

invalid_coordinates.show()

In [0]:
print("Total rows:", df_menu_items.count())
print("Distinct rows:", df_menu_items.distinct().count())

In [0]:
df_menu_items.select(
    *[
        sum(col(column).isNull().cast("int")).alias(column)
        for column in df_menu_items.columns
    ]
).show()

In [0]:
duplicate_item_ids = (
    df_menu_items
    .groupBy("item_id")
    .count()
    .filter(col("count") > 1)
)

print("Duplicate item IDs:", duplicate_item_ids.count())
duplicate_item_ids.show()

In [0]:
invalid_menu_restaurants = (
    df_menu_items
    .select("restaurant_id")
    .join(
        df_restaurants.select("restaurant_id"),
        on="restaurant_id",
        how="left_anti"
    )
)

print("Invalid restaurant IDs:", invalid_menu_restaurants.count())

invalid_menu_restaurants.show()

In [0]:
invalid_prices = df_menu_items.filter(
    col("price") <= 0
)

print("Menu items with price less than or equal to 0:", invalid_prices.count())

invalid_prices.show()

In [0]:
negative_prices = df_menu_items.filter(
    col("price") < 0
).count()

print("Negative prices:", negative_prices)

In [0]:
zero_prices = df_menu_items.filter(
    col("price") == 0
).count()

print("Zero prices:", zero_prices)

In [0]:
df_menu_items.filter(
    col("price") > 0
).select("price").summary().show()

In [0]:
from pyspark.sql.functions import when, col

df_menu_items_clean = df_menu_items.withColumn(
    "price",
    when(col("price") == 0, None)
    .otherwise(col("price"))
)

In [0]:
df_menu_items_clean.select(
    *[
        sum(col(column).isNull().cast("int")).alias(column)
        for column in df_menu_items_clean.columns
    ]
).show()

In [0]:
print("Total rows:", df_restaurants.count())
print("Distinct rows:", df_restaurants.distinct().count())

In [0]:
df_restaurants.select(
    *[
        sum(col(column).isNull().cast("int")).alias(column)
        for column in df_restaurants.columns
    ]
).show()

In [0]:
df_restaurants.filter(
    col("name").isNull()
).show(50, truncate=False)

In [0]:
from pyspark.sql.functions import col

suspicious_restaurants = df_restaurants.filter(
    col("name").isNull() &
    (col("rating") == 0) &
    (col("delivery_fee") == 0)
)

print(
    "Restaurants with NULL name, 0 rating, and 0 delivery fee:",
    suspicious_restaurants.count()
)

In [0]:
print(
    "Restaurants with 0 rating:",
    df_restaurants.filter(col("rating") == 0).count()
)

print(
    "Restaurants with 0 delivery fee:",
    df_restaurants.filter(col("delivery_fee") == 0).count()
)

In [0]:
df_restaurants.filter(
    col("rating") > 0
).select("rating").summary().show()

In [0]:
df_restaurants.filter(
    col("delivery_fee") > 0
).select("delivery_fee").summary().show()

In [0]:
from pyspark.sql.functions import when, col

df_restaurants_clean = (
    df_restaurants
    .withColumn(
        "rating",
        when(col("rating") == 0, None)
        .otherwise(col("rating"))
    )
    .withColumn(
        "delivery_fee",
        when(col("delivery_fee") == 0, None)
        .otherwise(col("delivery_fee"))
    )
)

In [0]:
df_restaurants_clean.select(
    *[
        sum(col(column).isNull().cast("int")).alias(column)
        for column in df_restaurants_clean.columns
    ]
).show()

In [0]:
duplicate_restaurant_ids = (
    df_restaurants_clean
    .groupBy("restaurant_id")
    .count()
    .filter(col("count") > 1)
)

print("Duplicate restaurant IDs:", duplicate_restaurant_ids.count())

In [0]:
invalid_restaurant_locations = (
    df_restaurants_clean
    .select("location_id")
    .join(
        df_locations.select("location_id"),
        on="location_id",
        how="left_anti"
    )
)

print("Invalid location IDs:", invalid_restaurant_locations.count())

invalid_restaurant_locations.show()

In [0]:
for name, df in {
    "customers": df_customers,
    "delivery_agents": df_delivery_agents,
    "locations": df_locations,
    "menu_items": df_menu_items,
    "restaurants": df_restaurants
}.items():
    print(name, "→", df.count())

In [0]:
from pyspark.sql.functions import col, count

duplicate_restaurant_ids = (
    df_restaurants
    .groupBy("restaurant_id")
    .count()
    .filter(col("count") > 1)
)

print("Duplicate restaurant IDs:", duplicate_restaurant_ids.count())

duplicate_restaurant_ids.show()

In [0]:
invalid_restaurant_locations = (
    df_restaurants
    .select("location_id")
    .join(
        df_locations.select("location_id"),
        on="location_id",
        how="left_anti"
    )
)

print(
    "Restaurant records with invalid location_id:",
    invalid_restaurant_locations.count()
)

invalid_restaurant_locations.show()

In [0]:
invalid_menu_restaurants = (
    df_menu_items
    .select("restaurant_id")
    .join(
        df_restaurants.select("restaurant_id"),
        on="restaurant_id",
        how="left_anti"
    )
)

print(
    "Menu items with invalid restaurant_id:",
    invalid_menu_restaurants.count()
)

invalid_menu_restaurants.show()

In [0]:
invalid_customer_locations = (
    df_customers
    .filter(col("location_id").isNotNull())
    .select("location_id")
    .join(
        df_locations.select("location_id"),
        on="location_id",
        how="left_anti"
    )
)

print(
    "Customers with invalid location_id:",
    invalid_customer_locations.count()
)

invalid_customer_locations.show()

In [0]:
orders_sample = spark.read.json(
    "/Volumes/workspace/dinedash/raw_data/orders/orders_2024_01.json"
)

orders_sample.printSchema()

display(orders_sample.limit(5))

In [0]:
df_orders = spark.read.json(
    "/Volumes/workspace/dinedash/raw_data/orders/*.json"
)

print("Total orders:", df_orders.count())
df_orders.printSchema()

In [0]:
df_orders.select(
    "_corrupt_record",
    "order_id"
).filter(
    df_orders["_corrupt_record"].isNotNull()
).show(10, truncate=False)

In [0]:
df_orders_clean_view = df_orders.select("*")

In [0]:
df_orders_clean_view.createOrReplaceTempView("orders_temp")

In [0]:
raw_orders = spark.read.text(
    "/Volumes/workspace/dinedash/raw_data/orders/orders_2024_01.json"
)

display(raw_orders.limit(5))

In [0]:
from pyspark.sql.functions import from_json, col

In [0]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    LongType,
    ArrayType
)

In [0]:
order_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("restaurant_id", StringType(), True),
    StructField("agent_id", StringType(), True),
    StructField("delivery_location_id", StringType(), True),

    StructField(
        "items_ordered",
        ArrayType(
            StructType([
                StructField("item_id", StringType(), True),
                StructField("item_name", StringType(), True),
                StructField("price", DoubleType(), True),
                StructField("quantity", LongType(), True)
            ])
        ),
        True
    ),

    StructField("total_amount", DoubleType(), True),
    StructField("tip", DoubleType(), True),
    StructField("payment_method", StringType(), True),
    StructField("order_status", StringType(), True)
])

In [0]:
parsed_orders = raw_orders.withColumn(
    "parsed_data",
    from_json(col("value"), order_schema)
)

In [0]:
parsed_orders.select(
    "value",
    "parsed_data"
).show(5, truncate=False)

In [0]:
valid_orders = parsed_orders.filter(
    col("parsed_data").isNotNull()
)

corrupt_orders = parsed_orders.filter(
    col("parsed_data").isNull()
)

print("Valid orders:", valid_orders.count())
print("Corrupt orders:", corrupt_orders.count())

In [0]:
df_orders_clean = valid_orders.select(
    "parsed_data.*"
)

df_orders_clean.printSchema()

In [0]:
from pyspark.sql.functions import sum, when

df_orders_clean.select(
    [
        sum(
            when(col(column).isNull(), 1).otherwise(0)
        ).alias(column)
        for column in df_orders_clean.columns
    ]
).show()

In [0]:
clean_orders = valid_orders.select(
    "parsed_data.*"
)

display(clean_orders.limit(5))

In [0]:
clean_orders.printSchema()

In [0]:
clean_orders.filter(
    col("delivery_location_id").isNull()
).show(5, truncate=False)

In [0]:
from pyspark.sql.functions import col, from_json, countDistinct

In [0]:
clean_orders.groupBy("customer_id") \
    .agg(
        countDistinct("delivery_location_id").alias("unique_locations"),
        count("*").alias("total_orders")
    ) \
    .orderBy(col("total_orders").desc()) \
    .show(10)

In [0]:
import pyspark.sql.functions as F

In [0]:
clean_orders.groupBy("customer_id") \
    .agg(
        F.countDistinct("delivery_location_id").alias("unique_locations"),
        F.count("*").alias("total_orders")
    ) \
    .orderBy(F.col("total_orders").desc()) \
    .show(10)

In [0]:
null_customers = clean_orders.filter(
    col("delivery_location_id").isNull()
).select("customer_id").distinct()

clean_orders.join(
    null_customers,
    on="customer_id",
    how="inner"
).groupBy("customer_id") \
 .agg(
     countDistinct("delivery_location_id").alias("unique_locations"),
     count("*").alias("total_orders")
 ) \
 .orderBy(col("total_orders").desc()) \
 .show(50)

In [0]:
clean_orders.groupBy("order_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

In [0]:
from pyspark.sql.functions import sum, when

clean_orders.select(
    sum(when(col("total_amount") < 0, 1).otherwise(0)).alias("negative_total_amount"),
    sum(when(col("tip") < 0, 1).otherwise(0)).alias("negative_tip")
).show()

In [0]:
from pyspark.sql.functions import explode, sum as spark_sum

order_amount_check = clean_orders \
    .select(
        "order_id",
        "total_amount",
        explode("items_ordered").alias("item")
    ) \
    .groupBy("order_id", "total_amount") \
    .agg(
        spark_sum(
            col("item.price") * col("item.quantity")
        ).alias("calculated_amount")
    )

order_amount_check.filter(
    col("total_amount") != col("calculated_amount")
).show(10, truncate=False)

In [0]:
from pyspark.sql.functions import round

order_amount_check.filter(
    round(col("total_amount"), 2) != round(col("calculated_amount"), 2)
).show(10, truncate=False)

In [0]:
from pyspark.sql.functions import explode, sum as spark_sum, when

items_check = clean_orders.select(
    "order_id",
    explode("items_ordered").alias("item")
)

items_check.select(
    spark_sum(
        when(col("item.price") < 0, 1).otherwise(0)
    ).alias("negative_prices"),
    
    spark_sum(
        when(col("item.quantity") <= 0, 1).otherwise(0)
    ).alias("invalid_quantities"),
    
    spark_sum(
        when(col("item.price") == 0, 1).otherwise(0)
    ).alias("zero_prices")
).show()

In [0]:
items_check.filter(
    col("item.price") == 0
).select(
    col("item.item_id").alias("item_id"),
    col("item.item_name").alias("item_name")
).groupBy(
    "item_id",
    "item_name"
).count().orderBy(
    col("count").desc()
).show(100, truncate=False)

In [0]:
zero_price_order_items = items_check.filter(
    col("item.price") == 0
).select(
    col("item.item_id").alias("item_id")
).distinct()

zero_price_menu_items = df_menu_items.filter(
    col("price") == 0
).select("item_id").distinct()

print(
    "Zero-price order items:",
    zero_price_order_items.count()
)

print(
    "Zero-price menu items:",
    zero_price_menu_items.count()
)

print(
    "Order zero-price items missing from menu zero-price items:",
    zero_price_order_items.subtract(zero_price_menu_items).count()
)

print(
    "Menu zero-price items missing from order zero-price items:",
    zero_price_menu_items.subtract(zero_price_order_items).count()
)

In [0]:
invalid_customer_ids = clean_orders.select("customer_id").distinct() \
    .join(
        df_customers.select("customer_id").distinct(),
        on="customer_id",
        how="left_anti"
    )

print("Order customer IDs not found in customers:", invalid_customer_ids.count())

invalid_customer_ids.show()

In [0]:
invalid_restaurant_ids = clean_orders.select("restaurant_id").distinct() \
    .join(
        df_restaurants.select("restaurant_id").distinct(),
        on="restaurant_id",
        how="left_anti"
    )

print("Order restaurant IDs not found in restaurants:", invalid_restaurant_ids.count())

invalid_restaurant_ids.show()

In [0]:
print([name for name in dir() if not name.startswith("_")])

In [0]:
invalid_agent_ids = clean_orders.select("agent_id").distinct() \
    .join(
        df_delivery_agents.select("agent_id").distinct(),
        on="agent_id",
        how="left_anti"
    )

print("Order agent IDs not found in delivery agents:", invalid_agent_ids.count())

invalid_agent_ids.show()

In [0]:
invalid_location_ids = clean_orders \
    .filter(col("delivery_location_id").isNotNull()) \
    .select("delivery_location_id").distinct() \
    .join(
        df_locations.select("location_id").distinct(),
        clean_orders["delivery_location_id"] == df_locations["location_id"],
        "left_anti"
    )

print("Order location IDs not found in locations:", invalid_location_ids.count())

invalid_location_ids.show()

In [0]:
print("Duplicate customers:", duplicate_customers.count())
print("Duplicate delivery agents:", duplicate_delivery_agents.count())
print("Duplicate restaurant IDs:", duplicate_restaurant_ids.count())
print("Duplicate location IDs:", duplicate_location_ids.count())
print("Duplicate item IDs:", duplicate_item_ids.count())

In [0]:
duplicate_customers.show(10, truncate=False)

In [0]:
df_customers_clean \
    .filter(col("customer_id") == "C5350") \
    .show(truncate=False)

In [0]:
df_customers_clean \
    .filter(col("customer_id") == "C5350") \
    .count()

In [0]:
df_customers \
    .filter(col("customer_id") == "C5350") \
    .count()

In [0]:
duplicate_delivery_agents.show(10, truncate=False)

In [0]:
df_delivery_agents \
    .filter(col("agent_id") == "A250") \
    .count()

In [0]:
df_delivery_agents_clean \
    .filter(col("agent_id") == "A250") \
    .count()

In [0]:
clean_orders \
    .filter(col("delivery_location_id").isNull()) \
    .select(
        "order_id",
        "customer_id",
        "restaurant_id",
        "agent_id",
        "delivery_location_id",
        "order_status"
    ) \
    .show(20, truncate=False)

In [0]:
missing_location_orders = clean_orders \
    .filter(col("delivery_location_id").isNull()) \
    .join(
        df_customers_clean.select(
            "customer_id",
            col("location_id").alias("customer_location_id")
        ),
        on="customer_id",
        how="left"
    )

missing_location_orders.select(
    "order_id",
    "customer_id",
    "customer_location_id",
    "delivery_location_id",
    "order_status"
).show(50, truncate=False)

In [0]:
missing_location_orders.select(
    "customer_location_id",
    "delivery_location_id"
).groupBy().agg(
    count("*").alias("total_missing_delivery_locations"),
    count("customer_location_id").alias("customers_with_location")
).show()

In [0]:
from pyspark.sql.functions import to_date, current_date

customer_date_check = df_customers_clean \
    .withColumn("dob_date", to_date("dob")) \
    .withColumn("signup_date_parsed", to_date("signup_date"))

customer_date_check.select(
    "customer_id",
    "dob",
    "signup_date"
).filter(
    (col("dob_date").isNull()) |
    (col("signup_date_parsed").isNull()) |
    (col("signup_date_parsed") < col("dob_date")) |
    (col("dob_date") > current_date()) |
    (col("signup_date_parsed") > current_date())
).show(50, truncate=False)

print(
    "Invalid/suspicious customer date records:",
    customer_date_check.filter(
        (col("dob_date").isNull()) |
        (col("signup_date_parsed").isNull()) |
        (col("signup_date_parsed") < col("dob_date")) |
        (col("dob_date") > current_date()) |
        (col("signup_date_parsed") > current_date())
    ).count()
)

In [0]:
from pyspark.sql.functions import datediff, current_date

In [0]:
customer_age_at_signup = df_customers_clean.withColumn(
    "age_at_signup",
    datediff(
        col("signup_date"),
        col("dob")
    ) / 365.25
)

customer_age_at_signup \
    .filter(col("age_at_signup") < 13) \
    .select(
        "customer_id",
        "dob",
        "signup_date",
        "age_at_signup"
    ) \
    .show(20, truncate=False)

In [0]:
customer_age_at_signup \
    .filter(col("age_at_signup") < 13) \
    .count()        

In [0]:
customer_age_at_signup \
    .selectExpr(
        "min(age_at_signup) as min_age",
        "max(age_at_signup) as max_age",
        "avg(age_at_signup) as avg_age"
    ) \
    .show()

In [0]:
customer_age_at_signup \
    .groupBy(
        when(col("age_at_signup") < 13, "Under 13")
        .when(col("age_at_signup") < 18, "13-17")
        .when(col("age_at_signup") < 25, "18-24")
        .when(col("age_at_signup") < 40, "25-39")
        .when(col("age_at_signup") < 60, "40-59")
        .otherwise("60+")
        .alias("age_group")
    ) \
    .count() \
    .orderBy("age_group") \
    .show()

In [0]:
df_restaurants_clean.select(
    "restaurant_id",
    "name",
    "location_id",
    "rating"
).summary(
    "count",
    "min",
    "max",
    "mean"
).show()

In [0]:
df_restaurants_clean.select(
    "restaurant_id",
    "name",
    "rating"
).filter(
    (col("rating") < 0) | (col("rating") > 5)
).show()

In [0]:
df_restaurants_clean.filter(
    (col("rating") < 0) | (col("rating") > 5)
).count()

In [0]:
df_delivery_agents_clean.filter(
    (col("rating") < 0) | (col("rating") > 5)
).count()

In [0]:
df_delivery_agents_clean.filter(
    col("phone").isNull()
).count()

In [0]:
from pyspark.sql.functions import sum, when, col

for name, dataframe in {
    "customers": df_customers_clean,
    "restaurants": df_restaurants_clean,
    "agents": df_delivery_agents_clean,
    "locations": df_locations,
    "menu_items": df_menu_items_clean,
    "orders": clean_orders
}.items():

    print(f"\n{name.upper()}")

    dataframe.select([
        sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in dataframe.columns
    ]).show(truncate=False)

In [0]:
df_customers_clean = df_customers_clean.fillna({
    "email": "Unknown"
})

In [0]:
df_restaurants_clean.filter(
    col("name").isNull() |
    col("rating").isNull() |
    col("delivery_fee").isNull()
).show(50, truncate=False)

In [0]:
df_restaurants_clean.select(
    F.avg("rating").alias("avg_rating"),
    F.expr("percentile_approx(rating, 0.5)").alias("median_rating"),
    F.avg("delivery_fee").alias("avg_delivery_fee"),
    F.expr("percentile_approx(delivery_fee, 0.5)").alias("median_delivery_fee")
).show()

In [0]:
df_restaurants_clean = (
    df_restaurants_clean
    .fillna({
        "name": "Unknown",
        "rating": 3.5,
        "delivery_fee": 2.9
    })
)

In [0]:
df_restaurants_clean.select(
    *[
        F.sum(col(c).isNull().cast("int")).alias(c)
        for c in df_restaurants_clean.columns
    ]
).show()

In [0]:
df_customers_clean = df_customers_clean.fillna({
    "email": "Unknown"
})

In [0]:
df_delivery_agents_clean = df_delivery_agents_clean.fillna({
    "phone": "Unknown"
})

In [0]:
median_price = df_menu_items_clean.select(
    F.expr("percentile_approx(price, 0.5)").alias("median")
).first()["median"]

print(median_price)

In [0]:
df_menu_items_clean = df_menu_items_clean.fillna({
    "price": median_price
})

In [0]:
def check_missing(df, name):
    print(f"\n{name}")
    
    df.select(
        *[
            F.sum(col(c).isNull().cast("int")).alias(c)
            for c in df.columns
        ]
    ).show()


check_missing(df_customers_clean, "CUSTOMERS")
check_missing(df_restaurants_clean, "RESTAURANTS")
check_missing(df_delivery_agents_clean, "DELIVERY AGENTS")
check_missing(df_locations, "LOCATIONS")
check_missing(df_menu_items_clean, "MENU ITEMS")
check_missing(df_orders_clean, "ORDERS")

In [0]:
print("Customers:", df_customers_clean.count())
print("Restaurants:", df_restaurants_clean.count())
print("Delivery Agents:", df_delivery_agents_clean.count())
print("Locations:", df_locations.count())
print("Menu Items:", df_menu_items_clean.count())
print("Orders:", df_orders_clean.count())

In [0]:
print("Duplicate customers:",
      df_customers_clean.groupBy("customer_id")
      .count()
      .filter(col("count") > 1)
      .count())

print("Duplicate delivery agents:",
      df_delivery_agents_clean.groupBy("agent_id")
      .count()
      .filter(col("count") > 1)
      .count())

print("Duplicate restaurants:",
      df_restaurants_clean.groupBy("restaurant_id")
      .count()
      .filter(col("count") > 1)
      .count())

print("Duplicate locations:",
      df_locations.groupBy("location_id")
      .count()
      .filter(col("count") > 1)
      .count())

print("Duplicate menu items:",
      df_menu_items_clean.groupBy("item_id")
      .count()
      .filter(col("count") > 1)
      .count())

In [0]:
# Check for orphan customer IDs in orders
df_orders_clean.join(
    df_customers_clean,
    on="customer_id",
    how="left_anti"
).count()

In [0]:
# Check for orphan restaurant IDs in orders
df_orders_clean.join(
    df_restaurants_clean,
    on="restaurant_id",
    how="left_anti"
).count()

In [0]:
# Check for orphan agent IDs in orders
df_orders_clean.join(
    df_delivery_agents_clean,
    on="agent_id",
    how="left_anti"
).count()

In [0]:
# Check delivery location IDs in orders that don't exist in locations
df_orders_clean.filter(
    col("delivery_location_id").isNotNull()
).join(
    df_locations,
    df_orders_clean["delivery_location_id"] == df_locations["location_id"],
    how="left_anti"
).count()

In [0]:
print("Missing customer locations:", 
      df_customers_clean.filter(col("location_id").isNull()).count())

print("Missing order delivery locations:", 
      df_orders_clean.filter(col("delivery_location_id").isNull()).count())